#### **DATA VALIDATION:**
---
Notebook for data quality for the initial data from GitHub.

In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
import pandas as pd

StatementMeta(, 6421be79-18e1-44f7-846f-ff77d9653845, 3, Finished, Available, Finished)

In [3]:
claims = spark.read.table("lk_atlas_insurance_data_BRONZE.dbo.claims_history")
applicants = spark.read.table("lk_atlas_insurance_data_BRONZE.dbo.insurance_applicants")
policies = spark.read.table("lk_atlas_insurance_data_BRONZE.dbo.insurance_policies")
payments = spark.read.table("lk_atlas_insurance_data_BRONZE.dbo.payment_history")

StatementMeta(, 6421be79-18e1-44f7-846f-ff77d9653845, 5, Finished, Available, Finished)

In [4]:
# Function to generate data quality report
def generate_quality_report(df, table_name):
    """Generate comprehensive data quality report for a DataFrame"""
    
    print(f"\n{'='*80}")
    print(f"DATA QUALITY REPORT: {table_name}")
    print(f"{'='*80}")
    
    # 1. Basic Information
    print(f"\n1. BASIC INFORMATION:")
    print(f"   Total Rows: {df.count():,}")
    print(f"   Total Columns: {len(df.columns)}")
    
    # 2. Schema Analysis
    print(f"\n2. SCHEMA ANALYSIS:")
    df.printSchema()
    
    # 3. Null Value Analysis
    print(f"\n3. NULL VALUE ANALYSIS:")
    null_counts = []
    for col in df.columns:
        null_count = df.filter(F.col(col).isNull()).count()
        total_count = df.count()
        null_pct = (null_count / total_count * 100) if total_count > 0 else 0
        null_counts.append({
            'column': col,
            'null_count': null_count,
            'null_percentage': null_pct,
            'dtype': str(df.schema[col].dataType)
        })
    
    null_df = spark.createDataFrame(null_counts)
    null_df.orderBy(F.desc('null_percentage')).show(truncate=False)
    
    # 4. Duplicate Analysis
    print(f"\n4. DUPLICATE ANALYSIS:")
    duplicate_counts = {}
    
    # Check for duplicate rows
    dup_rows = df.count() - df.distinct().count()
    print(f"   Duplicate rows: {dup_rows}")
    
    # Check key columns for duplicates
    key_columns = {
        'claims': ['Claim_ID', 'Customer_ID', 'Policy_Number'],
        'applicants': ['Customer_ID', 'ID_Number', 'Email'],
        'policies': ['Policy_Number', 'Applicant_ID'],
        'payments': ['Payment_ID', 'Policy_Number']
    }
    
    if table_name in key_columns:
        for key_col in key_columns[table_name]:
            if key_col in df.columns:
                dup_count = df.count() - df.select(key_col).distinct().count()
                if dup_count > 0:
                    print(f"   Duplicate {key_col}: {dup_count}")
    
    # 5. Data Type Inconsistencies
    print(f"\n5. DATA TYPE CHECKS:")
    
    # Check for invalid dates
    date_cols = [col for col in df.columns if 'date' in col.lower() or 'Date' in col]
    for date_col in date_cols:
        try:
            invalid_dates = df.filter(
                (F.col(date_col).isNotNull()) & 
                (F.col(date_col) > F.current_date())
            ).count()
            if invalid_dates > 0:
                print(f"   {date_col}: {invalid_dates} future dates")
        except:
            pass
    
    # Check numeric columns for negative values
    numeric_cols = [col for col in df.columns 
                   if any(num_type in str(df.schema[col].dataType).lower() 
                         for num_type in ['int', 'long', 'double', 'decimal', 'float'])]
    
    for num_col in numeric_cols:
        try:
            negative_count = df.filter(
                (F.col(num_col).isNotNull()) & 
                (F.col(num_col) < 0)
            ).count()
            if negative_count > 0:
                print(f"   {num_col}: {negative_count} negative values")
        except:
            pass
    
    # 6. Statistical Summary
    print(f"\n6. STATISTICAL SUMMARY (Numeric Columns):")
    if numeric_cols:
        df.select(*[F.col(c).cast('double') for c in numeric_cols]).summary().show()
    
    # 7. Categorical Value Analysis
    print(f"\n7. CATEGORICAL VALUE DISTRIBUTION (Sample):")
    string_cols = [col for col in df.columns if isinstance(df.schema[col].dataType, StringType)]
    
    for string_col in string_cols[:5]:  # Limit to first 5 to avoid too much output
        try:
            print(f"\n   Column: {string_col}")
            value_counts = df.groupBy(string_col).count().orderBy(F.desc('count')).limit(10)
            value_counts.show(truncate=False)
        except:
            pass
    
    return null_df

# Function to check referential integrity
def check_referential_integrity():
    """Check foreign key relationships between tables"""
    
    print(f"\n{'='*80}")
    print(f"REFERENTIAL INTEGRITY CHECKS")
    print(f"{'='*80}")
    
    # 1. Claims → Customers
    print(f"\n1. Claims referring to non-existent customers:")
    missing_customers = claims.join(
        applicants, 
        claims.Customer_ID == applicants.Customer_ID, 
        'left_anti'
    ).count()
    print(f"   Missing customers in claims: {missing_customers}")
    
    # 2. Claims → Policies
    print(f"\n2. Claims referring to non-existent policies:")
    missing_policies = claims.join(
        policies, 
        claims.Policy_Number == policies.Policy_Number, 
        'left_anti'
    ).count()
    print(f"   Missing policies in claims: {missing_policies}")
    
    # 3. Payments → Policies
    print(f"\n3. Payments referring to non-existent policies:")
    missing_policies_payments = payments.join(
        policies, 
        payments.Policy_Number == policies.Policy_Number, 
        'left_anti'
    ).count()
    print(f"   Missing policies in payments: {missing_policies_payments}")
    
    # 4. Policies → Customers
    print(f"\n4. Policies referring to non-existent customers:")
    missing_applicants = policies.join(
        applicants, 
        policies.Applicant_ID == applicants.Customer_ID, 
        'left_anti'
    ).count()
    print(f"   Missing customers in policies: {missing_applicants}")

# Function to check business rules
def check_business_rules():
    """Validate business logic rules"""
    
    print(f"\n{'='*80}")
    print(f"BUSINESS RULE VALIDATIONS")
    print(f"{'='*80}")
    
    # 1. Policy effective/expiration dates
    print(f"\n1. Invalid policy date ranges:")
    invalid_dates = policies.filter(
        policies.Expiration_Date < policies.Effective_Date
    ).count()
    print(f"   Policies with expiration before effective date: {invalid_dates}")
    
    # 2. Claim amount vs settlement amount
    print(f"\n2. Claim settlement logic:")
    # Settlement > Claim
    excess_settlement = claims.filter(
        (claims.Settlement_Amount.isNotNull()) &
        (claims.Settlement_Amount > claims.Claim_Amount)
    ).count()
    print(f"   Settlements exceeding claim amount: {excess_settlement}")
    
    # 3. Customer age validation
    print(f"\n3. Customer age anomalies:")
    age_issues = applicants.filter(
        (applicants.Age < 0) | 
        (applicants.Age > 120)
    ).count()
    print(f"   Invalid age values (outside 0-120): {age_issues}")
    
    # 4. Smoking status consistency
    print(f"\n4. Smoking data consistency:")
    smoking_inconsistency = applicants.filter(
        (applicants.Is_Smoker == 'Yes') & 
        (applicants.Previous_Smoker == 'Yes') &
        (applicants.Smoker_Years.isNull() | (applicants.Smoker_Years <= 0))
    ).count()
    print(f"   Current smokers missing smoker years: {smoking_inconsistency}")
    
    # 5. Payment amount validation
    print(f"\n5. Payment validation:")
    invalid_payments = payments.filter(
        payments.Amount_Paid <= 0
    ).count()
    print(f"   Invalid payment amounts (≤ 0): {invalid_payments}")
    
    # 6. Policy coverage validation
    print(f"\n6. Policy coverage issues:")
    coverage_issues = policies.filter(
        (policies.Coverage_Amount <= 0) |
        (policies.Premium_Amount <= 0)
    ).count()
    print(f"   Invalid coverage/premium amounts: {coverage_issues}")

# Function to check data consistency across tables
def check_cross_table_consistency():
    """Check consistency of data across multiple tables"""
    
    print(f"\n{'='*80}")
    print(f"CROSS-TABLE CONSISTENCY CHECKS")
    print(f"{'='*80}")
    
    # 1. Reinsurance company consistency
    print(f"\n1. Reinsurance company naming consistency:")
    
    # Check between claims and policies
    if 'Reinsurance_Company' in claims.columns and 'Reinsurance_Company' in policies.columns:
        claim_reinsurers = claims.select('Reinsurance_Company').distinct()
        policy_reinsurers = policies.select('Reinsurance_Company').distinct()
        
        print(f"   Unique reinsurers in claims: {claim_reinsurers.count()}")
        print(f"   Unique reinsurers in policies: {policy_reinsurers.count()}")
    
    # 2. Customer demographic consistency
    print(f"\n2. Customer demographic distribution:")
    if 'Gender' in applicants.columns:
        gender_dist = applicants.groupBy('Gender').count().orderBy(F.desc('count'))
        print("   Gender distribution in applicants:")
        gender_dist.show(truncate=False)
    
    # 3. Claim status distribution
    print(f"\n3. Claim status analysis:")
    if 'Status' in claims.columns:
        status_dist = claims.groupBy('Status').count().orderBy(F.desc('count'))
        print("   Claim status distribution:")
        status_dist.show(truncate=False)

# Function to generate summary report
def generate_summary_report():
    """Generate a summary data quality report"""
    
    print(f"\n{'='*80}")
    print(f"DATA QUALITY SUMMARY REPORT")
    print(f"{'='*80}")
    
    summary_data = []
    
    # Analyze each table
    tables = {
        'claims': claims,
        'applicants': applicants,
        'policies': policies,
        'payments': payments
    }
    
    for name, df in tables.items():
        total_rows = df.count()
        total_cols = len(df.columns)
        
        # Count nulls
        total_nulls = 0
        for col in df.columns:
            null_count = df.filter(F.col(col).isNull()).count()
            total_nulls += null_count
        
        # Count duplicates
        duplicate_rows = df.count() - df.distinct().count()
        
        summary_data.append({
            'Table': name.upper(),
            'Total Rows': f"{total_rows:,}",
            'Total Columns': total_cols,
            'Total Null Values': f"{total_nulls:,}",
            'Null %': f"{(total_nulls/(total_rows * total_cols) * 100):.2f}%",
            'Duplicate Rows': f"{duplicate_rows:,}",
            'Duplicate %': f"{(duplicate_rows/total_rows * 100):.2f}%" if total_rows > 0 else "0%"
        })
    
    # Create summary DataFrame
    summary_df = spark.createDataFrame(summary_data)
    summary_df.show(truncate=False)

# Execute all quality checks
print("🚀 STARTING COMPREHENSIVE DATA QUALITY CHECK")
print(f"{'='*80}")

# 1. Individual table reports
reports = {}
reports['claims'] = generate_quality_report(claims, 'CLAIMS')
reports['applicants'] = generate_quality_report(applicants, 'APPLICANTS')
reports['policies'] = generate_quality_report(policies, 'POLICIES')
reports['payments'] = generate_quality_report(payments, 'PAYMENTS')

# 2. Referential integrity checks
check_referential_integrity()

# 3. Business rule validations
check_business_rules()

# 4. Cross-table consistency
check_cross_table_consistency()

# 5. Summary report
generate_summary_report()

print(f"\n{'='*80}")
print(f"✅ DATA QUALITY CHECK COMPLETED")
print(f"{'='*80}")

# Additional: Save quality metrics to a table for tracking
def save_quality_metrics():
    """Save data quality metrics to a Delta table for monitoring"""
    
    quality_metrics = []
    current_timestamp = datetime.now()
    
    for table_name, report_df in reports.items():
        # Convert report to pandas for easier manipulation
        report_pd = report_df.toPandas()
        
        for _, row in report_pd.iterrows():
            quality_metrics.append({
                'check_timestamp': current_timestamp,
                'table_name': table_name,
                'column_name': row['column'],
                'null_count': int(row['null_count']),
                'null_percentage': float(row['null_percentage']),
                'data_type': row['dtype']
            })
    
    # Create metrics DataFrame
    metrics_df = spark.createDataFrame(quality_metrics)
    
    # Save to Delta table (uncomment if you want to persist)
    # metrics_df.write \
    #     .mode("append") \
    #     .format("delta") \
    #     .saveAsTable("lk_atlas_insurance_data_BRONZE.dbo.data_quality_metrics")
    
    print(f"\nQuality metrics collected for {len(quality_metrics)} columns")

# Optional: Save metrics
# save_quality_metrics()

StatementMeta(, 6421be79-18e1-44f7-846f-ff77d9653845, 6, Finished, Available, Finished)

🚀 STARTING COMPREHENSIVE DATA QUALITY CHECK

DATA QUALITY REPORT: CLAIMS

1. BASIC INFORMATION:
   Total Rows: 11,910
   Total Columns: 18

2. SCHEMA ANALYSIS:
root
 |-- Claim_ID: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Policy_Number: string (nullable = true)
 |-- Claim_Type: string (nullable = true)
 |-- Claim_Amount: long (nullable = true)
 |-- Date_of_Claim: timestamp (nullable = true)
 |-- Status: string (nullable = true)
 |-- Settlement_Amount: double (nullable = true)
 |-- Date_of_Settlement: timestamp (nullable = true)
 |-- Processing_Days: long (nullable = true)
 |-- Reinsurance: string (nullable = true)
 |-- Reinsurance_Type: string (nullable = true)
 |-- Reinsurer_Settlement: void (nullable = true)
 |-- Reinsurance_Company: string (nullable = true)
 |-- Claim_Handler: string (nullable = true)
 |-- Fraud_Indicators: string (nullable = true)
 |-- Complexity_Level: string (nullable = true)
 |-- Documentation_Status: string (nullable = true)


3. 

Py4JJavaError: An error occurred while calling o6656.count.
: java.lang.IllegalStateException: Couldn't find Reinsurer_Settlement#1056 in []
	at org.apache.spark.sql.catalyst.expressions.BindReferences$$anonfun$bindReference$1.applyOrElse(BoundAttribute.scala:80)
	at org.apache.spark.sql.catalyst.expressions.BindReferences$$anonfun$bindReference$1.applyOrElse(BoundAttribute.scala:73)
	at org.apache.spark.sql.catalyst.trees.TreeNode.$anonfun$transformDownWithPruning$1(TreeNode.scala:461)
	at org.apache.spark.sql.catalyst.trees.CurrentOrigin$.withOrigin(origin.scala:76)
	at org.apache.spark.sql.catalyst.trees.TreeNode.transformDownWithPruning(TreeNode.scala:461)
	at org.apache.spark.sql.catalyst.trees.TreeNode.$anonfun$transformDownWithPruning$3(TreeNode.scala:466)
	at org.apache.spark.sql.catalyst.trees.UnaryLike.mapChildren(TreeNode.scala:1216)
	at org.apache.spark.sql.catalyst.trees.UnaryLike.mapChildren$(TreeNode.scala:1215)
	at org.apache.spark.sql.catalyst.expressions.UnaryExpressi